# 03. 모델링 (Modeling)

---
### 0. 라이브러리 및 데이터 로드

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb

from sklearn import preprocessing
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

from sklearn.utils.class_weight import compute_sample_weight

import optuna

c:\Users\tyshi\Downloads\BDA학회원이탈률예측프로젝트\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
### 1. 데이터 로드

In [2]:
submission = pd.read_csv('./data/sample_submission.csv')
train = pd.read_csv('./data/train_processed.csv')
test = pd.read_csv('./data/test_processed.csv')
target_df = pd.read_csv('./data/train_target.csv')
test_id = pd.read_csv('./data/test_id.csv')

---
### 2. 전처리

#### X / Y 분리

In [3]:
X = train
y = target_df['withdrawal']

X = train.copy()
y = target_df['withdrawal'].copy()

# object 컬럼 있는지 확인
print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts(normalize=True))

X shape: (1056, 24)
y distribution:
withdrawal
1    0.691288
0    0.308712
Name: proportion, dtype: float64


#### train / valid 분리

In [4]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

target이 불균형이기 때문에, `stratify`를 적용

#### sample weight 계산

In [5]:
sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

---
### 3. 모델링

#### GBM

In [6]:
gbm = GradientBoostingClassifier(random_state=42)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'random_state': 42
    }

    model = GradientBoostingClassifier(**params)
    model.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = model.predict(X_valid)
    score = f1_score(y_valid, y_pred)

    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

gbm = GradientBoostingClassifier(**study.best_params, random_state=42)
gbm.fit(X_train, y_train, sample_weight=sample_weights)

y_pred = gbm.predict(X_valid)

print("Best Params:", study.best_params)
print("Best F1:", study.best_value)
print("Validation Accuracy:", accuracy_score(y_valid, y_pred))
print("Validation F1:", f1_score(y_valid, y_pred))

[I 2026-04-11 18:34:19,347] A new study created in memory with name: no-name-38a46f99-8fd0-49de-9edd-0cf87773f86a
[I 2026-04-11 18:34:19,643] Trial 0 finished with value: 0.6587301587301587 and parameters: {'n_estimators': 398, 'learning_rate': 0.043118525350339235, 'max_depth': 3, 'subsample': 0.7923173473323141, 'min_samples_split': 7, 'min_samples_leaf': 7, 'max_features': 'log2'}. Best is trial 0 with value: 0.6587301587301587.
[I 2026-04-11 18:34:19,853] Trial 1 finished with value: 0.5752212389380531 and parameters: {'n_estimators': 338, 'learning_rate': 0.01927626565749362, 'max_depth': 2, 'subsample': 0.6780883394735667, 'min_samples_split': 20, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6587301587301587.
[I 2026-04-11 18:34:20,524] Trial 2 finished with value: 0.7067137809187279 and parameters: {'n_estimators': 755, 'learning_rate': 0.18333938708647543, 'max_depth': 4, 'subsample': 0.8610164043070772, 'min_samples_split': 2, 'min_samples_leaf

Best Params: {'n_estimators': 439, 'learning_rate': 0.14039412702197054, 'max_depth': 5, 'subsample': 0.7679614090471711, 'min_samples_split': 12, 'min_samples_leaf': 6, 'max_features': 'log2'}
Best F1: 0.7254237288135593
Validation Accuracy: 0.6179245283018868
Validation F1: 0.7254237288135593


#### Random Forest

In [10]:
rf_model = RandomForestClassifier(random_state=42)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'random_state': 42
    }

    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = model.predict(X_valid)
    score = f1_score(y_valid, y_pred)

    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

rf_model = RandomForestClassifier(**study.best_params, random_state=42)
rf_model.fit(X_train, y_train, sample_weight=sample_weights)

y_pred = rf_model.predict(X_valid)

print("Best Params:", study.best_params)
print("Best F1:", study.best_value)
print("Validation Accuracy:", accuracy_score(y_valid, y_pred))
print("Validation F1:", f1_score(y_valid, y_pred))

[I 2026-04-11 18:35:50,829] A new study created in memory with name: no-name-5e5ec536-930b-4c56-b83e-71a97d4b452b
[I 2026-04-11 18:35:51,236] Trial 0 finished with value: 0.6075949367088608 and parameters: {'n_estimators': 409, 'max_depth': 30, 'min_samples_split': 15, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.6075949367088608.
[I 2026-04-11 18:35:51,999] Trial 1 finished with value: 0.6985294117647058 and parameters: {'n_estimators': 719, 'max_depth': 18, 'min_samples_split': 4, 'min_samples_leaf': 3}. Best is trial 1 with value: 0.6985294117647058.
[I 2026-04-11 18:35:52,133] Trial 2 finished with value: 0.6766917293233082 and parameters: {'n_estimators': 130, 'max_depth': 16, 'min_samples_split': 12, 'min_samples_leaf': 2}. Best is trial 1 with value: 0.6985294117647058.
[I 2026-04-11 18:35:52,600] Trial 3 finished with value: 0.6172839506172839 and parameters: {'n_estimators': 485, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 6}. Best is trial 1 with val

Best Params: {'n_estimators': 192, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 2}
Best F1: 0.7375886524822695
Validation Accuracy: 0.6509433962264151
Validation F1: 0.7375886524822695


#### AdaBoost

In [8]:
adaboost_model = AdaBoostClassifier(random_state=42)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.0, 1)
    }

    model = AdaBoostClassifier(**params)
    model.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = model.predict(X_valid)
    score = f1_score(y_valid, y_pred)

    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

adaboost_model = AdaBoostClassifier(**study.best_params, random_state=42)
adaboost_model.fit(X_train, y_train, sample_weight=sample_weights)

y_pred = adaboost_model.predict(X_valid)

print("Best Params:", study.best_params)
print("Best F1:", study.best_value)
print("Validation Accuracy:", accuracy_score(y_valid, y_pred))
print("Validation F1:", f1_score(y_valid, y_pred))

[I 2026-04-11 18:34:50,340] A new study created in memory with name: no-name-819ba45c-cce3-4a6c-b05b-5ec9ca050cee
[I 2026-04-11 18:34:50,548] Trial 0 finished with value: 0.42 and parameters: {'n_estimators': 162, 'learning_rate': 0.7831764205672302}. Best is trial 0 with value: 0.42.
[I 2026-04-11 18:34:51,103] Trial 1 finished with value: 0.3240223463687151 and parameters: {'n_estimators': 413, 'learning_rate': 0.024813582203992213}. Best is trial 0 with value: 0.42.
[I 2026-04-11 18:34:51,895] Trial 2 finished with value: 0.36649214659685864 and parameters: {'n_estimators': 588, 'learning_rate': 0.40772180766320365}. Best is trial 0 with value: 0.42.
[I 2026-04-11 18:34:53,158] Trial 3 finished with value: 0.5495495495495496 and parameters: {'n_estimators': 1000, 'learning_rate': 0.6649361021321011}. Best is trial 3 with value: 0.5495495495495496.
[I 2026-04-11 18:34:53,304] Trial 4 finished with value: 0.37037037037037035 and parameters: {'n_estimators': 110, 'learning_rate': 0.771

Best Params: {'n_estimators': 917, 'learning_rate': 0.903006990563359}
Best F1: 0.5882352941176471
Validation Accuracy: 0.5330188679245284
Validation F1: 0.5822784810126582


#### Light GBM

In [9]:
lgb_model = lgb.LGBMClassifier(n_estimators=100, random_state=42,verbose=-1)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt', 'dart']),
        'learning_rate': trial.suggest_float('learning_rate', 0.1, 0.3),
        'max_depth': trial.suggest_int('max_depth', 0, 20),
        'num_leaves': trial.suggest_int('num_leaves', 5, 70),
        'random_state': 42,
        'verbose': -1
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = model.predict(X_valid)
    score = f1_score(y_valid, y_pred)

    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

lgb_model = lgb.LGBMClassifier(**study.best_params, random_state=42, verbose= -1)
lgb_model.fit(X_train, y_train, sample_weight=sample_weights)

y_pred = lgb_model.predict(X_valid)

print("Best Params:", study.best_params)
print("Best F1:", study.best_value)
print("Validation Accuracy:", accuracy_score(y_valid, y_pred))
print("Validation F1:", f1_score(y_valid, y_pred))

[I 2026-04-11 18:35:20,013] A new study created in memory with name: no-name-512589ba-0d87-4270-b504-264ea1b89533
[I 2026-04-11 18:35:22,931] Trial 0 finished with value: 0.602510460251046 and parameters: {'n_estimators': 248, 'boosting_type': 'gbdt', 'learning_rate': 0.23859142292060934, 'max_depth': 1, 'num_leaves': 46}. Best is trial 0 with value: 0.602510460251046.
[I 2026-04-11 18:35:23,501] Trial 1 finished with value: 0.7152777777777778 and parameters: {'n_estimators': 419, 'boosting_type': 'dart', 'learning_rate': 0.22740547689647195, 'max_depth': 10, 'num_leaves': 32}. Best is trial 1 with value: 0.7152777777777778.
[I 2026-04-11 18:35:23,911] Trial 2 finished with value: 0.7177700348432056 and parameters: {'n_estimators': 334, 'boosting_type': 'dart', 'learning_rate': 0.24466367068631278, 'max_depth': 10, 'num_leaves': 47}. Best is trial 2 with value: 0.7177700348432056.
[I 2026-04-11 18:35:24,025] Trial 3 finished with value: 0.6285714285714286 and parameters: {'n_estimators

Best Params: {'n_estimators': 267, 'boosting_type': 'gbdt', 'learning_rate': 0.29689256929206553, 'max_depth': 14, 'num_leaves': 56}
Best F1: 0.7440273037542662
Validation Accuracy: 0.6462264150943396
Validation F1: 0.7440273037542662
